# 📊 Notebook 02 — Statistical Impact Analysis
## Issaquah Creek Salmon Return Study | Summer 2025

This notebook:
1. Multiple regression: which stressors predict Issaquah Creek returns?
2. Partial dependence analysis — isolating each factor
3. Lag analysis — how many years back do snowpack and ocean conditions matter?
4. Stressor importance ranking — the 'what matters most' table for the awareness report
5. Pre/post 2000 structural break analysis

In [ ]:
import sys; sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from pathlib import Path

from src.features import build_features

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
FIG_DIR = Path('../outputs/figures')

df = build_features(pd.read_csv('../data/processed/issaquah_creek_master.csv'))
print(f'Loaded: {df.shape[0]} years × {df.shape[1]} columns')

## 1. Multiple Linear Regression — Chinook Returns

In [ ]:
# Select predictor variables for regression
predictors = [
    'swe_apr1_in',          # April 1 snowpack
    'min_summer_flow_cfs',  # summer low flow
    'max_summer_temp_c',    # summer peak water temperature
    'pdo_lag1_winter',      # ocean conditions prior year
    'impervious_pct',       # urban development
    'days_above_18c',       # thermal stress days
]

target = 'chinook_total'
avail_preds = [p for p in predictors if p in df.columns]
model_df = df[avail_preds + [target]].dropna()

print(f'Rows available for regression: {len(model_df)}')
print(f'Predictors used: {avail_preds}')

In [ ]:
# Check multicollinearity (VIF > 5 is concerning)
X_vif = sm.add_constant(model_df[avail_preds])
vif_data = pd.DataFrame({
    'Feature': avail_preds,
    'VIF': [variance_inflation_factor(X_vif.values, i+1)
            for i in range(len(avail_preds))]
}).sort_values('VIF', ascending=False)

print('Variance Inflation Factors (VIF):')
print('  VIF < 5 = acceptable | VIF 5-10 = moderate | VIF > 10 = high multicollinearity')
print(vif_data.to_string(index=False))

In [ ]:
# OLS regression with statsmodels for full significance reporting
X = sm.add_constant(model_df[avail_preds])
y = model_df[target]

ols_model = sm.OLS(y, X).fit()
print(ols_model.summary())

In [ ]:
# Coefficient plot — which predictors are significant and in which direction?
coef = ols_model.params.drop('const')
conf = ols_model.conf_int().drop('const')
pvals = ols_model.pvalues.drop('const')

# Standardize coefficients for comparison
std_coef = coef * model_df[avail_preds].std() / y.std()

name_map = {
    'swe_apr1_in':         'April 1 Snowpack',
    'min_summer_flow_cfs': 'Min Summer Streamflow',
    'max_summer_temp_c':   'Max Summer Water Temp',
    'pdo_lag1_winter':     'PDO Index (1-yr lag)',
    'impervious_pct':      'Impervious Surface %',
    'days_above_18c':      'Days Above 18°C',
}

plot_df = pd.DataFrame({
    'feature':  [name_map.get(k, k) for k in std_coef.index],
    'std_coef': std_coef.values,
    'significant': pvals.values < 0.05,
}).sort_values('std_coef')

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#1F4E79' if s else '#AAAAAA' for s in plot_df['significant']]
bars = ax.barh(plot_df['feature'], plot_df['std_coef'], color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Standardized Coefficient\n(positive = more fish, negative = fewer fish)', fontsize=10)
ax.set_title('Chinook Return Predictors — Regression Coefficients\n'
             'Blue = statistically significant (p < 0.05), Gray = not significant',
             fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(FIG_DIR / '05_regression_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Lag Analysis — Does Prior-Year Snowpack Matter?

In [ ]:
# Test different SWE and PDO lag combinations to find the best predictors
lag_results = []

for swe_lag in [0, 1, 2]:
    for pdo_lag in [1, 2, 3]:
        swe_col = 'swe_apr1_in' if swe_lag == 0 else f'swe_apr1_in_lag{swe_lag}'
        pdo_col = f'pdo_winter_mean_lag{pdo_lag}' if f'pdo_winter_mean_lag{pdo_lag}' in df.columns else 'pdo_lag1_winter'

        if swe_col not in df.columns or pdo_col not in df.columns:
            continue

        combo_df = df[['chinook_total', swe_col, pdo_col, 'impervious_pct']].dropna()
        if len(combo_df) < 15:
            continue

        X_lag = sm.add_constant(combo_df[[swe_col, pdo_col, 'impervious_pct']])
        m = sm.OLS(combo_df['chinook_total'], X_lag).fit()
        lag_results.append({
            'swe_lag': swe_lag, 'pdo_lag': pdo_lag,
            'r2': m.rsquared, 'adj_r2': m.rsquared_adj, 'n': len(combo_df)
        })

lag_df = pd.DataFrame(lag_results).sort_values('adj_r2', ascending=False)
print('Lag structure comparison (higher adj R² = better fit):')
print(lag_df.to_string(index=False))
best = lag_df.iloc[0]
print(f'\n✓ Best lag: SWE lag-{best["swe_lag"]}, PDO lag-{best["pdo_lag"]}  (adj R² = {best["adj_r2"]:.3f})')

## 3. Scatter Plots — Returns vs. Top Predictors

In [ ]:
scatter_pairs = [
    ('swe_apr1_in',        'April 1 Snowpack (in)',    '#4472C4'),
    ('min_summer_flow_cfs','Min Summer Streamflow (cfs)','#4472C4'),
    ('impervious_pct',     'Impervious Surface (%)',    '#7030A0'),
    ('max_summer_temp_c',  'Max Summer Water Temp (°C)','#C00000'),
]
scatter_pairs = [(c, l, col) for c, l, col in scatter_pairs if c in df.columns]

n = len(scatter_pairs)
fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
if n == 1: axes = [axes]

for ax, (xcol, xlabel, color) in zip(axes, scatter_pairs):
    plot_df = df[['water_year', xcol, 'chinook_total']].dropna()
    ax.scatter(plot_df[xcol], plot_df['chinook_total'],
               c=plot_df['water_year'], cmap='Blues', s=50, zorder=3)

    # Regression line
    slope, intercept, r, p, _ = stats.linregress(plot_df[xcol], plot_df['chinook_total'])
    x_line = np.linspace(plot_df[xcol].min(), plot_df[xcol].max(), 100)
    ax.plot(x_line, slope * x_line + intercept, 'r-', linewidth=1.5)

    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel('Chinook Returns', fontsize=10)
    ax.set_title(f'r = {r:.2f}  {sig}', fontsize=10)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.suptitle('Chinook Returns vs. Key Stressors (Issaquah Creek)\n'
             'Color = year (darker = more recent)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / '06_scatter_stressors.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Stressor Impact Summary Table

In [ ]:
# Build ranked summary table for awareness report
stressor_info = {
    'April 1 Snowpack':        ('swe_apr1_in',        'positive', 'Snowpack feeds summer baseflow; low SWE = warm, low-flow streams'),
    'Summer Low Flow':         ('min_summer_flow_cfs', 'positive', 'Low summer flows concentrate fish, raise temps, limit passage'),
    'Peak Water Temperature':  ('max_summer_temp_c',   'negative', 'Temps above 18°C are thermal stress; above 21°C can be lethal'),
    'PDO Ocean Index (lag 1)': ('pdo_lag1_winter',     'negative', 'Warm-phase PDO reduces marine food supply during ocean years'),
    'Impervious Surface':      ('impervious_pct',      'negative', 'Urban runoff = pollution, flashy flows, fine sediment in redds'),
    'Days Above 18°C':         ('days_above_18c',      'negative', 'Duration of thermal stress compounds peak temperature impacts'),
}

summary_rows = []
for stressor_name, (col, direction, mechanism) in stressor_info.items():
    if col not in df.columns:
        continue
    valid = df[['chinook_total', col]].dropna()
    r, p = stats.spearmanr(valid['chinook_total'], valid[col])
    sig = '*** p<0.001' if p < 0.001 else ('** p<0.01' if p < 0.01 else ('* p<0.05' if p < 0.05 else 'ns'))
    trend_valid = df[col].dropna()
    trend_slope = np.polyfit(range(len(trend_valid)), trend_valid, 1)[0]

    summary_rows.append({
        'Stressor': stressor_name,
        'Correlation (r)': round(r, 3),
        'Significance': sig,
        '40-yr Trend': f"{'+' if trend_slope > 0 else ''}{trend_slope:.3f}/yr",
        'Mechanism': mechanism,
    })

impact_df = pd.DataFrame(summary_rows)
impact_df = impact_df.sort_values('Correlation (r)', key=abs, ascending=False)
impact_df.insert(0, 'Rank', range(1, len(impact_df)+1))

print('STRESSOR IMPACT RANKING — Issaquah Creek Chinook Returns')
print('=' * 90)
print(impact_df[['Rank','Stressor','Correlation (r)','Significance','40-yr Trend']].to_string(index=False))

# Save for awareness report
impact_df.to_csv('../outputs/stressor_impact_ranking.csv', index=False)
print('\nSaved → outputs/stressor_impact_ranking.csv')

## 5. Key Statistical Findings

*(Fill in after running on real data)*

**Regression model fit:** R² = TBD  
**Best predictors (p < 0.05):** TBD  
**Best lag structure:** SWE lag-TBD, PDO lag-TBD  

| Stressor | Direction | Significance | Local Trend |
|----------|-----------|--------------|-------------|
| Snowpack | Positive | TBD | Declining |
| Summer Low Flow | Positive | TBD | Declining |
| Water Temp | Negative | TBD | Increasing |
| PDO | Negative | TBD | Variable |
| Impervious % | Negative | TBD | Increasing |

**➡ Proceed to `03_modeling.ipynb` for predictive model and scenario projections.**